In [ ]:
import duckdb, pandas as pd, numpy as np
con = duckdb.connect("../data/processed/flights.duckdb")
def q(sql): return con.execute(sql).df()

print(q("""
SELECT date_trunc('month', flight_date) AS month, COUNT(*) AS flights
FROM flights GROUP BY 1 ORDER BY 1
""").to_string(index=False))



     month  flights
2025-01-01   522269
2025-02-01   496476
2025-03-01   592301
2025-04-01   577730
2025-05-01   597574
2025-06-01   599472


In [5]:
con.execute(f"""
CREATE OR REPLACE TABLE route_hist AS
SELECT Origin, Dest,
       COUNT(*) AS route_flights,
       AVG(ArrDel15) AS route_delay_rate,
       AVG(ArrDelay) AS route_mean_delay
FROM flights WHERE flight_date < DATE '{CUTOFF}'
GROUP BY 1,2 HAVING COUNT(*) >= 50
""")

con.execute(f"""
CREATE OR REPLACE TABLE carrier_hist AS
SELECT carrier, AVG(ArrDel15) AS carrier_delay_rate
FROM flights WHERE flight_date < DATE '{CUTOFF}' GROUP BY 1
""")

con.execute(f"""
CREATE OR REPLACE TABLE airport_hist AS
SELECT Origin AS airport,
       AVG(ArrDel15) AS origin_delay_rate,
       COUNT(*) * 1.0 / COUNT(DISTINCT flight_date) AS avg_daily_departures
FROM flights WHERE flight_date < DATE '{CUTOFF}' GROUP BY 1
""")

con.execute(f"""
CREATE OR REPLACE TABLE airport_hour_hist AS
SELECT Origin AS airport, CRSDepTime // 100 AS hour,
       COUNT(*) * 1.0 / COUNT(DISTINCT flight_date) AS departures_this_hour,
       AVG(ArrDel15) AS hour_delay_rate
FROM flights WHERE flight_date < DATE '{CUTOFF}' GROUP BY 1,2
""")

for t in ["route_hist", "carrier_hist", "airport_hist", "airport_hour_hist"]:
    print(t, q(f"SELECT COUNT(*) AS n FROM {t}").iloc[0,0])

route_hist 4914
carrier_hist 14
airport_hist 333
airport_hour_hist 3894


In [10]:
con.execute(open("../sql/sql1_Features.sql").read())
print(q("SELECT COUNT(*) AS rows FROM features"))
print(q("SELECT * FROM features LIMIT 5").to_string())

      rows
0  3385822
  flight_date carrier Origin Dest    tail  dep_hour  dep_minute  day_of_week  month  Distance  CRSElapsedTime  prev_arr_delay  is_first_leg  leg_number  turnaround_min  route_delay_rate  route_mean_delay  carrier_delay_rate  origin_delay_rate  origin_daily_departures  dest_delay_rate  departures_this_hour  hour_delay_rate  target  arr_delay_min
0  2025-05-22      WN    MSY  HOU  N8884Q        17          30            4      5     302.0            80.0            -4.0             0           5              45          0.172043          1.866935            0.171499           0.193485               147.779661         0.169331             11.550847         0.274395     0.0           -2.0
1  2025-05-22      WN    HOU  PHX  N8884Q        19          35            4      5    1020.0           170.0            -2.0             0           6              45          0.162162          1.972973            0.171499           0.169331               142.478992         0.164545

In [11]:
print(q("""
SELECT
  COUNT(*) AS n,
  SUM(CASE WHEN turnaround_min < 0 THEN 1 ELSE 0 END) AS negative,
  MIN(turnaround_min) AS min_val,
  MEDIAN(turnaround_min) AS median_val,
  MAX(turnaround_min) AS max_val
FROM features WHERE turnaround_min IS NOT NULL
""").to_string(index=False))

      n  negative  min_val  median_val  max_val
2530912   42195.0     -625        55.0     1435


In [12]:
con.execute("""
UPDATE features
SET turnaround_min = NULL
WHERE turnaround_min < 0 OR turnaround_min > 720
""")
print(q("SELECT COUNT(*) AS nulls FROM features WHERE turnaround_min IS NULL"))

    nulls
0  907194


In [13]:
cols = q("SELECT * FROM features LIMIT 1").columns.tolist()
drop = ["target", "arr_delay_min", "flight_date", "tail"]
feat_cols = [c for c in cols if c not in drop]
print(len(feat_cols), "features:")
for c in feat_cols:
    print(" -", c)

21 features:
 - carrier
 - Origin
 - Dest
 - dep_hour
 - dep_minute
 - day_of_week
 - month
 - Distance
 - CRSElapsedTime
 - prev_arr_delay
 - is_first_leg
 - leg_number
 - turnaround_min
 - route_delay_rate
 - route_mean_delay
 - carrier_delay_rate
 - origin_delay_rate
 - origin_daily_departures
 - dest_delay_rate
 - departures_this_hour
 - hour_delay_rate


In [14]:
print(q("""
SELECT target,
       COUNT(*) AS n,
       AVG(prev_arr_delay) AS avg_prev_delay,
       AVG(turnaround_min) AS avg_turnaround,
       AVG(leg_number) AS avg_leg,
       AVG(dep_hour) AS avg_hour,
       AVG(route_delay_rate) AS avg_route_rate
FROM features GROUP BY 1
""").to_string(index=False))

 target       n  avg_prev_delay  avg_turnaround  avg_leg  avg_hour  avg_route_rate
    1.0  741313       29.235819       79.182964 3.249039 14.488410        0.210214
    0.0 2644509       -3.927832       74.301276 2.745792 12.554245        0.192564


In [15]:
print(q("""
SELECT
  SUM(CASE WHEN is_first_leg = 1 THEN 1 ELSE 0 END) AS first_legs,
  SUM(CASE WHEN is_first_leg = 0 AND turnaround_min IS NULL THEN 1 ELSE 0 END) AS clipped,
  COUNT(*) AS total
FROM features
""").to_string(index=False))

 first_legs  clipped   total
   854910.0  52284.0 3385822


In [16]:
con.close()